# Handling Missing Data in Pandas

**Topic:** Data cleaning with `isna()`, `notna()`, `dropna()`, `fillna()`, `ffill()`, and `bfill()`  
**Level:** Beginner  
**Goal:** Learn how to identify missing values and choose an appropriate strategy for handling them.

---

## 1. What is missing data?

Missing data occurs when a value is not available for a particular row or column. In Pandas, missing values are commonly represented by `NaN`, `None`, or `pd.NA`, depending on the data type.

Missing values may appear because:
- a customer did not provide information;
- a sensor or system failed to record a value;
- data was not available at the time of collection;
- different files were combined and some fields did not match.

**Important:** A missing value is not the same as zero, an empty string, or the text `"NA"`. Those may be valid values unless your data-cleaning rules say otherwise.

### Example dataset

We will create a small sales dataset directly in this notebook, so the examples can run without downloading a separate CSV file.

In [1]:
import pandas as pd
import numpy as np

sales = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005, 1006],
    "customer_name": ["Aarav", "Priya", "Rohan", None, "Vikram", "Ananya"],
    "category": ["Electronics", "Home", "Electronics", "Stationery", None, "Electronics"],
    "product": ["Laptop", "Office Chair", "Wireless Mouse", "Notebook Pack", "Desk Lamp", "Keyboard"],
    "quantity": [1, 2, 3, 5, np.nan, 1],
    "sales_channel": ["Online", None, "Store", "Online", "Store", None]
})

sales

,order_id,customer_name,category,product,quantity,sales_channel
0,1001,Aarav,Electronics,Laptop,1.0,Online
1,1002,Priya,Home,Office Chair,2.0,NaN
2,1003,Rohan,Electronics,Wireless Mouse,3.0,Store
3,1004,NaN,Stationery,Notebook Pack,5.0,Online
4,1005,Vikram,NaN,Desk Lamp,NaN,Store
5,1006,Ananya,Electronics,Keyboard,1.0,NaN


## 2. Detect missing values

Before changing a dataset, inspect where values are missing.

### `isna()` / `isnull()`

These methods return `True` where a value is missing and `False` otherwise. `isna()` and `isnull()` are equivalent; use one consistently.

- `df.isna()` — shows a True/False result for every cell.
- `df.isna().sum()` — counts missing values in each column.
- `df.isna().sum().sum()` — counts all missing cells in the DataFrame.
- `df.isna().any()` — tells you which columns contain at least one missing value.

### `notna()`

Returns `True` for values that are present and `False` for missing values.

In [2]:
# Missing-value mask
sales.isna()

,order_id,customer_name,category,product,quantity,sales_channel
0,False,False,False,False,False,False
1,False,False,False,False,False,True
2,False,False,False,False,False,False
3,False,True,False,False,False,False
4,False,False,True,False,True,False
5,False,False,False,False,False,True


In [3]:
# Number of missing values in each column
sales.isna().sum()

order_id         0
customer_name    1
category         1
product          0
quantity         1
sales_channel    2
dtype: int64

In [4]:
# Total number of missing cells
sales.isna().sum().sum()

np.int64(5)

In [5]:
# Which columns contain any missing values?
sales.isna().any()

order_id         False
customer_name     True
category          True
product          False
quantity          True
sales_channel     True
dtype: bool

In [6]:
# Rows where customer_name is present
sales[sales["customer_name"].notna()]

,order_id,customer_name,category,product,quantity,sales_channel
0,1001,Aarav,Electronics,Laptop,1.0,Online
1,1002,Priya,Home,Office Chair,2.0,NaN
2,1003,Rohan,Electronics,Wireless Mouse,3.0,Store
4,1005,Vikram,NaN,Desk Lamp,NaN,Store
5,1006,Ananya,Electronics,Keyboard,1.0,NaN


## 3. Remove missing data with `dropna()`

Use `dropna()` when removing rows or columns is appropriate. By default, it returns a new DataFrame and removes every row containing at least one missing value. The original DataFrame is unchanged unless you assign the result or use `inplace=True`.

### Common parameters

| Syntax | Meaning |
|---|---|
| `df.dropna()` | Drop rows with one or more missing values (default: `axis=0`, `how="any"`). |
| `df.dropna(axis=0)` | Drop rows containing missing values. |
| `df.dropna(axis=1)` | Drop columns containing missing values. |
| `df.dropna(how="all")` | Drop rows only when every value in the row is missing. |
| `df.dropna(subset=["quantity"])` | Check missing values only in the specified column(s). |
| `df.dropna(thresh=4)` | Keep rows with at least 4 non-missing values. |

**Caution:** `axis=1` can remove a whole column if it contains even one missing value. In real datasets, this may discard useful information.

In [7]:
# Remove rows that contain at least one missing value
sales.dropna()

,order_id,customer_name,category,product,quantity,sales_channel
0,1001,Aarav,Electronics,Laptop,1.0,Online
2,1003,Rohan,Electronics,Wireless Mouse,3.0,Store


In [8]:
# Remove columns that contain at least one missing value
sales.dropna(axis=1)

,order_id,product
0,1001,Laptop
1,1002,Office Chair
2,1003,Wireless Mouse
3,1004,Notebook Pack
4,1005,Desk Lamp
5,1006,Keyboard


In [9]:
# Drop rows only when every column in the row is missing
sales.dropna(how="all")

,order_id,customer_name,category,product,quantity,sales_channel
0,1001,Aarav,Electronics,Laptop,1.0,Online
1,1002,Priya,Home,Office Chair,2.0,NaN
2,1003,Rohan,Electronics,Wireless Mouse,3.0,Store
3,1004,NaN,Stationery,Notebook Pack,5.0,Online
4,1005,Vikram,NaN,Desk Lamp,NaN,Store
5,1006,Ananya,Electronics,Keyboard,1.0,NaN


In [10]:
# Require quantity to be present, while allowing missing values elsewhere
sales.dropna(subset=["quantity"])

,order_id,customer_name,category,product,quantity,sales_channel
0,1001,Aarav,Electronics,Laptop,1.0,Online
1,1002,Priya,Home,Office Chair,2.0,NaN
2,1003,Rohan,Electronics,Wireless Mouse,3.0,Store
3,1004,NaN,Stationery,Notebook Pack,5.0,Online
5,1006,Ananya,Electronics,Keyboard,1.0,NaN


In [11]:
# Keep rows with at least 4 non-missing values
sales.dropna(thresh=4)

,order_id,customer_name,category,product,quantity,sales_channel
0,1001,Aarav,Electronics,Laptop,1.0,Online
1,1002,Priya,Home,Office Chair,2.0,NaN
2,1003,Rohan,Electronics,Wireless Mouse,3.0,Store
3,1004,NaN,Stationery,Notebook Pack,5.0,Online
4,1005,Vikram,NaN,Desk Lamp,NaN,Store
5,1006,Ananya,Electronics,Keyboard,1.0,NaN


### Does `dropna()` change the original DataFrame?

By default, no. The following creates a cleaned DataFrame and leaves `sales` unchanged:

```python
clean_sales = sales.dropna()
```

You can also assign the result back:

```python
sales = sales.dropna()
```

`inplace=True` changes the existing DataFrame directly, but assignment is often easier to read and maintain:

```python
sales.dropna(inplace=True)
```

Avoid dropping rows automatically without considering why the data is missing. Removing too many records can introduce bias or reduce the usefulness of the dataset.

## 4. Replace missing values with `fillna()`

`fillna()` replaces missing values with a value you choose. The replacement should make sense for the column and the problem you are solving.

### A. Fill with one value

This applies the same replacement to every missing cell. It is useful for demonstration, but a single value such as `"Unknown"` is not appropriate for every column—for example, it would turn a numeric column into mixed or text data.

In [12]:
# Use a separate copy so the original data remains available
sales_fill = sales.copy()

# Demonstration only: same replacement for every missing cell
sales_fill.fillna("Unknown")

,order_id,customer_name,category,product,quantity,sales_channel
0,1001,Aarav,Electronics,Laptop,1.0,Online
1,1002,Priya,Home,Office Chair,2.0,Unknown
2,1003,Rohan,Electronics,Wireless Mouse,3.0,Store
3,1004,Unknown,Stationery,Notebook Pack,5.0,Online
4,1005,Vikram,Unknown,Desk Lamp,Unknown,Store
5,1006,Ananya,Electronics,Keyboard,1.0,Unknown


### B. Fill selected columns with a dictionary

Pass a dictionary to use different replacement values for different columns. This is usually clearer because each column can have its own rule.

- Use a label such as `"Unknown"` for a categorical/text column when appropriate.
- Use a numeric value for a numeric column only when that value is meaningful.
- Do not replace missing quantities with `0` unless missing truly means zero.

In [13]:
sales_fill = sales.copy()

sales_fill.fillna({
    "customer_name": "Unknown",
    "category": "Uncategorized",
    "sales_channel": "Not recorded",
    "quantity": sales["quantity"].median()
})

,order_id,customer_name,category,product,quantity,sales_channel
0,1001,Aarav,Electronics,Laptop,1.0,Online
1,1002,Priya,Home,Office Chair,2.0,Not recorded
2,1003,Rohan,Electronics,Wireless Mouse,3.0,Store
3,1004,Unknown,Stationery,Notebook Pack,5.0,Online
4,1005,Vikram,Uncategorized,Desk Lamp,2.0,Store
5,1006,Ananya,Electronics,Keyboard,1.0,Not recorded


### C. Fill with a statistic: mean, median, or mode

For numeric columns, common options include:
- **Mean:** the arithmetic average; can be affected by extreme values.
- **Median:** the middle value; often more robust to outliers.
- **Mode:** the most frequent value; often used for categorical data.

Choose based on the data distribution and business meaning. Imputation can affect analysis, so it is good practice to document the rule used.

In [14]:
# Median is calculated from the available quantity values
quantity_median = sales["quantity"].median()

sales_median_filled = sales.copy()
sales_median_filled["quantity"] = sales_median_filled["quantity"].fillna(quantity_median)

sales_median_filled

,order_id,customer_name,category,product,quantity,sales_channel
0,1001,Aarav,Electronics,Laptop,1.0,Online
1,1002,Priya,Home,Office Chair,2.0,NaN
2,1003,Rohan,Electronics,Wireless Mouse,3.0,Store
3,1004,NaN,Stationery,Notebook Pack,5.0,Online
4,1005,Vikram,NaN,Desk Lamp,2.0,Store
5,1006,Ananya,Electronics,Keyboard,1.0,NaN


### D. Limit the number of values filled

`limit` restricts how many missing values are filled consecutively in each column when using `fillna()` with a scalar value.

```python
df.fillna("Unknown", limit=1)
```

This fills at most one missing value per column in the relevant run; it does **not** mean “fill only one missing cell in the entire DataFrame.”

In [15]:
sales.copy().fillna("Unknown", limit=1)

,order_id,customer_name,category,product,quantity,sales_channel
0,1001,Aarav,Electronics,Laptop,1.0,Online
1,1002,Priya,Home,Office Chair,2.0,Unknown
2,1003,Rohan,Electronics,Wireless Mouse,3.0,Store
3,1004,Unknown,Stationery,Notebook Pack,5.0,Online
4,1005,Vikram,Unknown,Desk Lamp,Unknown,Store
5,1006,Ananya,Electronics,Keyboard,1.0,NaN


## 5. Forward fill and backward fill

These methods copy nearby observed values into missing positions. They are especially useful for ordered data such as time series, but the row order must be meaningful.

### Forward fill: `ffill()`

Fills a missing value using the previous available value in the selected direction.

```python
df.ffill()
```

### Backward fill: `bfill()`

Fills a missing value using the next available value in the selected direction.

```python
df.bfill()
```

By default, filling happens vertically down each column (`axis=0`). Use `axis=1` to fill horizontally across columns. Horizontal filling is only sensible when adjacent columns have compatible meanings; it is usually inappropriate for unrelated fields such as product name and quantity.

**Caution:** Forward/backward filling can carry a value into a row where it does not belong. For example, copying a previous customer's name into another order would be incorrect.

In [16]:
# Forward fill: use the previous value in each column
sales.ffill()

,order_id,customer_name,category,product,quantity,sales_channel
0,1001,Aarav,Electronics,Laptop,1.0,Online
1,1002,Priya,Home,Office Chair,2.0,Online
2,1003,Rohan,Electronics,Wireless Mouse,3.0,Store
3,1004,Rohan,Stationery,Notebook Pack,5.0,Online
4,1005,Vikram,Stationery,Desk Lamp,5.0,Store
5,1006,Ananya,Electronics,Keyboard,1.0,Store


In [17]:
# Backward fill: use the next value in each column
sales.bfill()

,order_id,customer_name,category,product,quantity,sales_channel
0,1001,Aarav,Electronics,Laptop,1.0,Online
1,1002,Priya,Home,Office Chair,2.0,Store
2,1003,Rohan,Electronics,Wireless Mouse,3.0,Store
3,1004,Vikram,Stationery,Notebook Pack,5.0,Online
4,1005,Vikram,Electronics,Desk Lamp,1.0,Store
5,1006,Ananya,Electronics,Keyboard,1.0,NaN


In [18]:
# Horizontal forward fill (demonstration only; not suitable for these unrelated sales columns)
sales.ffill(axis=1)

,order_id,customer_name,category,product,quantity,sales_channel
0,1001,Aarav,Electronics,Laptop,1.0,Online
1,1002,Priya,Home,Office Chair,2.0,2.0
2,1003,Rohan,Electronics,Wireless Mouse,3.0,Store
3,1004,1004,Stationery,Notebook Pack,5.0,Online
4,1005,Vikram,Vikram,Desk Lamp,Desk Lamp,Store
5,1006,Ananya,Electronics,Keyboard,1.0,1.0


## 6. Verify the result

After cleaning, check whether missing values remain and compare the data before and after cleaning.

In [19]:
clean_sales = sales.copy()
clean_sales["customer_name"] = clean_sales["customer_name"].fillna("Unknown")
clean_sales["category"] = clean_sales["category"].fillna("Uncategorized")
clean_sales["sales_channel"] = clean_sales["sales_channel"].fillna("Not recorded")
clean_sales["quantity"] = clean_sales["quantity"].fillna(clean_sales["quantity"].median())

print("Missing values before cleaning:")
print(sales.isna().sum())

print("\\nMissing values after cleaning:")
print(clean_sales.isna().sum())

Missing values before cleaning:
order_id         0
customer_name    1
category         1
product          0
quantity         1
sales_channel    2
dtype: int64
\nMissing values after cleaning:
order_id         0
customer_name    0
category         0
product          0
quantity         0
sales_channel    0
dtype: int64


## 7. Best practices

1. **Inspect first:** use `isna().sum()` before choosing a cleaning method.
2. **Understand why values are missing:** do not assume every missing value should be deleted or replaced.
3. **Choose a column-specific rule:** numeric, categorical, and date columns often need different approaches.
4. **Avoid arbitrary replacements:** values such as `"python"` or `0` may distort analysis if they have no real meaning.
5. **Preserve the original data:** use `.copy()` or assign the cleaned result to a new variable while learning and experimenting.
6. **Check the result:** count missing values again and verify data types when necessary.
7. **Document your decision:** explain why rows were removed or how replacement values were selected.

## 8. Quick reference

| Goal | Pandas expression |
|---|---|
| Find missing cells | `df.isna()` |
| Count missing values per column | `df.isna().sum()` |
| Count all missing cells | `df.isna().sum().sum()` |
| Keep rows where a column is present | `df[df["column"].notna()]` |
| Drop rows with any missing value | `df.dropna()` |
| Drop columns with any missing value | `df.dropna(axis=1)` |
| Drop rows where all values are missing | `df.dropna(how="all")` |
| Require a particular column | `df.dropna(subset=["column"])` |
| Keep rows with at least N present values | `df.dropna(thresh=N)` |
| Replace missing values | `df.fillna(value)` |
| Forward fill | `df.ffill()` |
| Backward fill | `df.bfill()` |

---

**Key takeaway:** There is no single best method for missing data. Choose between dropping, constant/statistical imputation, and forward/backward filling based on the meaning of the data and the goal of your analysis.